In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/deep-past-initiative-machine-translation/train.csv
/kaggle/input/deep-past-initiative-machine-translation/test.csv
/kaggle/input/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/deep-past-initiative-machine-translation/resources.csv
/kaggle/input/predictions1/preds.json
/kaggle/input/byt5-small-new/byt5_akkadian2/final/config.json
/kaggle/input/byt5-small-new/byt5_akkadian2/final/training_args.bin
/kaggle/input/byt5-small-new/byt5_akkadian2/final/tokenizer_config.json
/kaggle/input/byt5-small-ne

In [2]:
import json
import pandas as pd
import math

In [3]:
test_df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/test.csv")
test_df.head()

,id,text_id,line_start,line_end,transliteration
0,0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,3,332fda50,25,30,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


In [4]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    print(dirname)

/kaggle/input
/kaggle/input/deep-past-initiative-machine-translation
/kaggle/input/predictions1
/kaggle/input/byt5-small-new
/kaggle/input/byt5-small-new/byt5_akkadian2
/kaggle/input/byt5-small-new/byt5_akkadian2/final


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [6]:
MODEL_PATH = "/kaggle/input/byt5-small-new/byt5_akkadian2/final"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

2026-01-26 08:35:10.634876: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769416510.812457      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769416510.863464      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769416511.281240      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769416511.281294      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769416511.281297      24 computation_placer.cc:177] computation placer alr

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [9]:
model.to(device)
model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(384, 1472)
  (encoder): T5Stack(
    (embed_tokens): Embedding(384, 1472)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1472, out_features=384, bias=False)
              (k): Linear(in_features=1472, out_features=384, bias=False)
              (v): Linear(in_features=1472, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=1472, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1472, out_features=3584, bias=False)
              (wi_1): Linear(in_features=1472, out_features=3584, bias=False)
              (w

In [10]:
preds = []
for src in test_df["transliteration"]:
    src = src.replace("translate akkadian to english:\n", "").strip()

    inputs = tokenizer(src, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            early_stopping=True
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds.append(pred)

In [11]:
preds

['-bi„-ma mup-pu-um aa a-lim(ki) i-lí-kam-ar-ra-tim qí... da-ni kà-a\na-wa-bar\naa-qi-il… ay-ip-ri-ne ka-a-r-sa-ha-mu-aa\nga-fa-va-xa-za-ca-ja-o-lu-ki-mi-ká-a?\naA-qI-ILKAM-AR-RA-TIM',
 '-na mup-pì-im aa a-lim(ki) ia-ra-tí-au kà-ni-ia i-lá-qé\nKÙ-NA MUP-PÌ-IM AA A-LIM(KI) IA-RA-TÍ-AU KÀ-NI-IA I-LÁ-QÉ\nKI Ia-Ra„-Im Aa A“-né-mì...\nKi Iá A LIm(Ki)\nKY ICH',
 'im lu-up-ta-áa-me-a-ni-a.gal-lim-i-dí-in-lu té-ra-at ú-kà-lá i-na muP-pì-im\nLU-UP-TA-ÁA-ME-A-NI A.GAL-LIM I-DÍ-IN LU TA Ú-KÀ-LÁ IM\nLu-Up-Ta-Ca-Ni-bi„-it a.GaL-lym im',
 '-lá KÙ.AN lu a-na DAM.GÀR-ru-tim i-dí-in au-mì a wi-lim-lu-up-ta-+nim\nau me-+e-er muP-pì-ni-a kà-ar-ma ú wa-ba-r-ra-tIm aé-bi„-la\na-wi li?\nan LU-UP-TA-NIM\naÉ-BI-LIM-LU']

In [12]:
import pandas as pd

submission = pd.DataFrame({
    "id": test_df["id"],      
    "translation": preds
})

In [13]:
submission.to_csv("submission.csv", index=False)